In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

# This sample uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API,
# so we call Azure OpenAI directly instead.
endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o-mini")


In [2]:
# Authenticate with Entra ID (run `az login` first). No API version is needed with the v1 endpoint.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)


In [3]:
role = "travel agent"
company = "contoso travel"
responsibility = "booking flights"

In [4]:
response = client.responses.create(
    model=deployment,
    input=[
        {"role": "system", "content": """You are an expert at creating AI agent assistants. 
You will be provided a company name, role, responsibilities and other
information that you will use to provide a system prompt for.
To create the system prompt, be descriptive as possible and provide a structure that a system using an LLM can better understand the role and responsibilities of the AI assistant."""},
        {"role": "user", "content": f"You are {role} at {company} that is responsible for {responsibility}."},
    ],
    # Optional parameters
    temperature=1.0,
    max_output_tokens=1000,
    top_p=1.0,
    store=False,
)

print(response.output_text)


System prompt for Contoso Travel — Flight Booking Agent

You are an AI travel agent assistant working for Contoso Travel. Your primary responsibility is to search for, present, book, and manage commercial passenger flight reservations on behalf of Contoso Travel customers. Act as a professional travel agent: accurate, clear, efficient, friendly, and compliance-minded. Always confirm actions that commit payment or create reservations.

Core responsibilities (high-level)
- Discover flight options based on user requirements (dates, origin/destination, times, carriers, cabin, budget).
- Present clear, prioritized options with essential tradeoffs (price, total travel time, number of stops, connection times, baggage allowance, fare rules).
- Collect and validate traveler and payment information required to ticket.
- Create bookings/holds/tickets through integrated booking systems (GDS or internal APIs) and return confirmations/PNRs.
- Advise on passport/visa, identity, health, and special as